# Review, export, and optionally publish the trained Transformer

This notebook is intentionally separate from training. It loads the averaged v2 checkpoint, builds a self-contained Hugging Face bundle, verifies that bundle locally, and uploads only when `publish_to_hub = True`. Authenticate with `hf auth login` or `HF_TOKEN`; never paste a token into a cell.

In [ ]:
import importlib
import json
from pathlib import Path
import shutil
import sys

from huggingface_hub import HfApi
from safetensors.torch import save_model
from tokenizers import Tokenizer
import torch

repo_root = Path.cwd().resolve()
while not (repo_root / "pyproject.toml").exists():
    if repo_root.parent == repo_root:
        raise FileNotFoundError("Run this notebook from inside the llm-recipes repository")
    repo_root = repo_root.parent

component_dir = repo_root / "notebooks/00_transformer_components"
sys.path.insert(0, str(component_dir))
from train_transformer import Transformer  # noqa: E402

checkpoint_path = repo_root / "checkpoints/transformer_en_de/averaged.pt"
tokenizer_path = repo_root / "data/tokenizers/wmt14_bpe_en_de.json"
export_dir = checkpoint_path.parent / "hub_export"
hub_model_name = "transformer-en-de-wmt14"
private_repo = False
publish_to_hub = False

## 1. Review the selected checkpoint

Confirm the update, epoch, validation loss, and evaluation predictions before publishing. The upload flag remains disabled by default.

In [ ]:
if not checkpoint_path.exists():
    raise FileNotFoundError(f"Checkpoint not found: {checkpoint_path}")
checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=False, mmap=True)
print(f"Checkpoint: {checkpoint_path}")
print(f"Update: {checkpoint['update']:,}; epoch: {checkpoint['epoch'] + 1}")
print(f"Best validation loss: {checkpoint['best_validation_loss']:.4f}")
print(f"Split: {checkpoint.get('split', {}).get('strategy', 'legacy row-random')}")

predictions_path = repo_root / "evaluations/transformer_en_de/newstest2014_beam4.jsonl"
if predictions_path.exists():
    print("\nSaved evaluation samples:")
    with predictions_path.open(encoding="utf-8") as predictions_file:
        for _, line in zip(range(10), predictions_file):
            record = json.loads(line)
            print(f"EN: {record['source']}\nREF: {record['target']}\nDE: {record['prediction']}\n")
else:
    print(f"No saved predictions at {predictions_path}; run evaluate_transformer.py first.")

## 2. Build a self-contained bundle

The export contains Safetensors weights, tokenizer and model configuration, model/inference Python files, and a model card with loading instructions.

In [ ]:
tokenizer = Tokenizer.from_file(str(tokenizer_path))
model = Transformer(**checkpoint["config"])
model.load_state_dict(checkpoint["model"])
model.eval()

export_dir.mkdir(parents=True, exist_ok=True)
save_model(model, str(export_dir / "model.safetensors"))
shutil.copy2(tokenizer_path, export_dir / "tokenizer.json")
shutil.copy2(component_dir / "train_transformer.py", export_dir / "modeling_transformer.py")
shutil.copy2(component_dir / "hub_inference.py", export_dir / "hub_inference.py")
dataset_sources = """# Training data sources\n\n- Europarl v7\n- Common Crawl WMT13\n- News Commentary v9\n\nValidation: newstest2013. Test: newstest2014. Review each WMT-distributed corpus's terms before redistribution.\n"""
(export_dir / "DATASET_SOURCES.md").write_text(dataset_sources)

config = {
    "architecture": "Transformer",
    "model_config": checkpoint["config"],
    "source_language": "en",
    "target_language": "de",
    "training_updates": checkpoint["update"],
    "training_epoch": checkpoint["epoch"] + 1,
    "best_validation_loss": checkpoint["best_validation_loss"],
    "split": checkpoint.get("split"),
}
(export_dir / "config.json").write_text(json.dumps(config, indent=2) + "\n")

model_card = f"""---
language: [en, de]
pipeline_tag: translation
library_name: pytorch
tags: [transformer, translation, from-scratch]
license: other
---
# Transformer English-German (WMT14)

An educational, from-scratch encoder-decoder Transformer trained with the WMT14 English-German recipe. The exported weights average the retained validation-best checkpoints and achieved best validation loss {checkpoint['best_validation_loss']:.4f}.

Training sources: Europarl v7, Common Crawl, and News Commentary v9; validation is newstest2013 and test is newstest2014. See DATASET_SOURCES.md and the upstream WMT corpus terms. The implementation code is Apache-2.0.

## Load and translate

```python
from huggingface_hub import snapshot_download
import sys

model_dir = snapshot_download("YOUR_NAMESPACE/{hub_model_name}")
sys.path.insert(0, model_dir)
from hub_inference import TransformerTranslator

translator = TransformerTranslator.from_pretrained(model_dir)
print(translator.translate("How are you?"))
```

Dependencies: PyTorch, tokenizers, safetensors, tqdm.
"""
(export_dir / "README.md").write_text(model_card)
print(f"Exported bundle to {export_dir}")

## 3. Verify the exported bundle locally

In [ ]:
sys.path.insert(0, str(export_dir))
import modeling_transformer
import hub_inference
importlib.reload(modeling_transformer)
importlib.reload(hub_inference)
translator = hub_inference.TransformerTranslator.from_pretrained(export_dir)
for source, translation in zip(
    ["How are you?", "I like learning languages."],
    translator.translate(["How are you?", "I like learning languages."]),
):
    print(f"{source} -> {translation}")

## 4. Optional upload

Only after reviewing the metrics, saved predictions, and local translations above, set `publish_to_hub = True` in the configuration cell and rerun this cell.

In [ ]:
if publish_to_hub:
    api = HfApi()
    username = api.whoami()["name"]
    repo_id = f"{username}/{hub_model_name}"
    api.create_repo(repo_id, private=private_repo, exist_ok=True)
    api.upload_folder(
        repo_id=repo_id,
        folder_path=export_dir,
        commit_message="Upload selected English-German Transformer",
    )
    print(f"Uploaded to https://huggingface.co/{repo_id}")
else:
    print("Upload disabled. Set publish_to_hub = True only when satisfied with the model.")